# Deep Learning in Asset Pricing
**Chen, Pelger & Zhu (2019)** — ArXivist Reproduction Notebook

This notebook walks through the full pipeline:
1. Data setup and preprocessing
2. Model construction (GAN + LSTM)
3. Training (3-step adversarial procedure)
4. Evaluation: Sharpe Ratio, Explained Variation, Cross-Sectional R²
5. Variable importance and SDF structure analysis

> **ArXivist confidence**: 0.85 overall | See `sir.json` for ambiguities.

---

### ⚠ Data Note
This notebook uses **synthetic data** for demonstration. For paper results, configure real CRSP paths in `configs/config.yaml`.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
import json

from dlap.utils.config import load_config, set_seed, get_device
from dlap.models.gan_model import GANAssetPricingModel
from dlap.training.trainer import GANTrainer
from dlap.training.losses import MomentConditionLoss, UnconditionalMomentLoss
from dlap.evaluation.metrics import (
    compute_sharpe_ratio, compute_explained_variation,
    compute_xs_r2, compute_variable_importance, compute_all_metrics
)
from dlap.data.dataset import make_synthetic_dataset

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Configuration
Load hyperparameters from `configs/config.yaml`. Optimal values from Table I of the paper.

In [ ]:
cfg = load_config('../configs/config.yaml')

print('Key hyperparameters (Table I):')
m = cfg['model']
t = cfg['training']
print(f"  SDF layers (HL):       {m['sdf_num_layers']}")
print(f"  SDF hidden units (HU): {m['sdf_hidden_units']}")
print(f"  SDF macro states (SMV):{m['sdf_macro_states']}")
print(f"  Cond macro states:     {m['cond_macro_states']}")
print(f"  Cond moments (CHU):    {m['cond_hidden_units']}")
print(f"  Learning rate:         {t['learning_rate']}")
print(f"  Dropout (DR):          {t['dropout']}")
print(f"  Ensemble size:         {t['num_ensemble_models']}")

## 2. Data
Synthetic panel matching the simulation setup from Section IV (N=500, T=600, SR≈1).

**For real results**: replace with CRSP data loaded as described in `data/README_data.md`.

In [ ]:
# Synthetic data (Section IV simulation setup)
dataset = make_synthetic_dataset(T=600, N=500, seed=42, device=device)
print(dataset)

macro, chars, returns, panel_weights = dataset.get_all()
print(f'macro:   {macro.shape}   (T=600, 178 macro variables)')
print(f'chars:   {chars.shape}  (T=600, N=500, 46 firm characteristics)')
print(f'returns: {returns.shape}  (T=600, N=500 excess returns)')
print(f'panel_weights: {panel_weights.shape}  (T_i/T per stock)')

# Paper split: 20yr train / 5yr valid / 25yr test
train_data = (macro[:250], chars[:250], returns[:250])
valid_data = (macro[250:350], chars[250:350], returns[250:350])
test_data  = (macro[350:], chars[350:], returns[350:])

## 3. Model Architecture
Build the full GAN asset pricing model from the SIR.

In [ ]:
model = GANAssetPricingModel(cfg).to(device)
print(model)

sdf_params = sum(p.numel() for p in model.sdf_parameters())
cond_params = sum(p.numel() for p in model.conditional_parameters())
print(f'\nSDF network parameters:         {sdf_params:,}')
print(f'Conditional network parameters: {cond_params:,}')
print(f'Total trainable parameters:     {sdf_params + cond_params:,}')

## 4. Forward Pass Verification
Verify tensor shapes match the SIR specification.

In [ ]:
model.eval()
with torch.no_grad():
    macro_b = macro[:250].unsqueeze(0)  # [1, T, 178]
    omega, F_t, M_t, h_t = model.forward_sdf(macro_b, chars[:250], returns[:250])
    g, h_t_g = model.forward_conditional(macro_b, chars[:250])
    beta = model.forward_loadings(h_t, chars[:250])

print('Tensor shape verification (SIR spec):')
print(f'  omega:  {tuple(omega.shape)}  expect [T=250, N=500]          ✓' if omega.shape == (250, 500) else f'  omega:  {tuple(omega.shape)}  MISMATCH')
print(f'  F_t:    {tuple(F_t.shape)}  expect [T=250]                   ✓' if F_t.shape == (250,) else f'  F_t: MISMATCH')
print(f'  M_t:    {tuple(M_t.shape)}  expect [T=250]                   ✓' if M_t.shape == (250,) else f'  M_t: MISMATCH')
print(f'  h_t:    {tuple(h_t.shape)}  expect [T=250, 4]                ✓' if h_t.shape == (250, 4) else f'  h_t: MISMATCH')
print(f'  g:      {tuple(g.shape)}  expect [T=250, N=500, 8]           ✓' if g.shape == (250, 500, 8) else f'  g: MISMATCH')
print(f'  h_t_g:  {tuple(h_t_g.shape)}  expect [T=250, 32]             ✓' if h_t_g.shape == (250, 32) else f'  h_t_g: MISMATCH')
print(f'  beta:   {tuple(beta.shape)}  expect [T=250, N=500]           ✓' if beta.shape == (250, 500) else f'  beta: MISMATCH')

# Verify SDF identity: M_t = 1 - F_t
assert torch.allclose(M_t, 1.0 - F_t), 'SDF identity violated!'
print(f'\nSDF identity M_t = 1 - F_t: ✓')
print(f'F_t mean: {F_t.mean():.4f}, std: {F_t.std():.4f}')

## 5. Training
Run the 3-step GAN training procedure (Section III.D). Using 5 epochs here for demonstration.

In [ ]:
import os
os.makedirs('../checkpoints', exist_ok=True)
os.makedirs('../results', exist_ok=True)

trainer = GANTrainer(model, cfg, device)

# Quick demo: 5 epochs (set debug=False and increase max_epochs for full training)
history = trainer.fit(
    train_data=train_data,
    valid_data=valid_data,
    max_epochs=5,
    patience=20,
    log_every=1,
    debug=False,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_sr'], label='Train SR', color='steelblue')
axes[0].plot(history['valid_sr'], label='Valid SR', color='orange')
axes[0].set_title('Sharpe Ratio During Training')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Sharpe Ratio (monthly)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['loss_sdf'], color='crimson')
axes[1].set_title('SDF Network Loss (Eq. 3)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Moment Condition Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Evaluation
Compute all three metrics (Table III equivalent).

In [ ]:
model.eval()
results = {}

for split_name, (macro_s, chars_s, ret_s) in [
    ('train', train_data),
    ('valid', valid_data),
    ('test', test_data),
]:
    macro_b = macro_s.unsqueeze(0)
    with torch.no_grad():
        omega, F_t, M_t, h_t = model.forward_sdf(macro_b, chars_s, ret_s)
        beta = model.forward_loadings(h_t, chars_s)
    metrics = compute_all_metrics(ret_s, beta, F_t, annualize=True)
    results[split_name] = metrics

print('='*65)
print(f'{"":<10} {"SR (ann)":>12} {"EV":>12} {"XS-R2":>12}')
print('-'*65)
for split, m in results.items():
    print(f'{split:<10} {m["sharpe_ratio"]:>12.4f} {m["explained_variation"]:>12.4f} {m["xs_r2"]:>12.4f}')
print('='*65)
print()
print('Paper Table III (GAN, monthly SR annualized):')
print(f'  Train SR: 2.68 → ann ≈ 9.3 | EV: 0.20 | XS-R2: 0.12')
print(f'  Valid SR: 1.43 → ann ≈ 5.0 | EV: 0.09 | XS-R2: 0.01')
print(f'  Test SR:  0.75 → ann ≈ 2.6 | EV: 0.08 | XS-R2: 0.23')

## 7. LSTM Hidden States
Plot the 4 macroeconomic hidden state processes (Figure 15 equivalent).

In [ ]:
model.eval()
with torch.no_grad():
    macro_full_b = macro.unsqueeze(0)
    h_states, _ = model.sdf_macro_rnn(macro_full_b)
    h_states = h_states.squeeze(0).cpu().numpy()  # [T, 4]

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
colors = ['steelblue', 'crimson', 'forestgreen', 'darkorange']

for i, ax in enumerate(axes):
    ax.scatter(range(len(h_states)), h_states[:, i], s=4, color=colors[i], alpha=0.7)
    ax.set_ylabel(f'Macro_{i}', fontsize=10)
    ax.grid(True, alpha=0.2)
    ax.axhline(0, color='black', linewidth=0.5)

axes[-1].set_xlabel('Time (months)')
fig.suptitle('Macroeconomic Hidden State Processes (LSTM outputs)\n'
             'cf. Figure 15 in Chen, Pelger & Zhu (2019)', fontsize=12)
plt.tight_layout()
plt.savefig('../results/macro_hidden_states.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Variable Importance
Compute characteristic importance via average absolute gradient (Figure 13 equivalent).

In [ ]:
CHAR_NAMES = [
    'ST_REV', 'SUV', 'r12_2', 'NOA', 'SGA2S', 'LME', 'RNA', 'LTurnover',
    'Lev', 'Resid_Var', 'ROA', 'E2P', 'D2P', 'Spread', 'CF2P', 'BEME',
    'Variance', 'D2A', 'PCM', 'A2ME', 'AT', 'Rel2High', 'CF', 'Q',
    'Investment', 'PM', 'DPI2A', 'ROE', 'S2P', 'FC2Y', 'AC', 'CTO',
    'LT_Rev', 'OP', 'PROF', 'IdioVol', 'r12_7', 'Beta', 'OA', 'ATO',
    'MktBeta', 'OL', 'C', 'r36_13', 'NI', 'r2_1',
]

macro_test_b = test_data[0].unsqueeze(0)
vi = compute_variable_importance(model, macro_test_b.squeeze(0), test_data[1], test_data[2], CHAR_NAMES)

# Sort and plot top 20
vi_sorted = sorted(vi.items(), key=lambda x: x[1], reverse=True)
names = [x[0] for x in vi_sorted[:20]]
scores = [x[1] for x in vi_sorted[:20]]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(names[::-1], scores[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Average Absolute Gradient (normalized)', fontsize=11)
ax.set_title('Top 20 Characteristic Importance — GAN\ncf. Figure 13 in Chen, Pelger & Zhu (2019)', fontsize=12)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/variable_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nPaper top-3: ST_REV, SUV, r12_2 (Short-Term Reversal, Unexplained Volume, Momentum)')

## 9. SDF Factor Cumulative Returns
Plot cumulative excess returns of the SDF factor (Figure 7 equivalent).

In [ ]:
model.eval()
with torch.no_grad():
    macro_all_b = macro.unsqueeze(0)
    omega_all, F_t_all, M_t_all, _ = model.forward_sdf(macro_all_b, chars, returns)

F_np = F_t_all.cpu().numpy()
F_norm = F_np / F_np.std()  # normalize by std (paper: each factor normalized by std)
cumret = np.cumsum(F_norm)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cumret, color='steelblue', linewidth=1.5, label='GAN SDF Factor')
ax.axvline(250, color='grey', linestyle='--', alpha=0.5, label='Train/Valid split')
ax.axvline(350, color='orange', linestyle='--', alpha=0.5, label='Valid/Test split')
ax.set_xlabel('Months')
ax.set_ylabel('Cumulative Excess Return (std-normalized)')
ax.set_title('SDF Factor Cumulative Returns\ncf. Figure 7 in Chen, Pelger & Zhu (2019)')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('../results/sdf_cumulative_returns.png', dpi=120, bbox_inches='tight')
plt.show()

sr_full = compute_sharpe_ratio(F_t_all, annualize=True)
print(f'Full-sample annual Sharpe Ratio: {sr_full.item():.4f}')
print(f'Paper reports: GAN test annual SR ≈ 2.6')

## 10. Ambiguity Notes
Documenting low-confidence implementation choices that should be verified.

In [ ]:
with open('../../../sir-registry/paper_deep_learning_asset_pricing/sir.json') as f:
    sir = json.load(f)

print('Implementation ambiguities from SIR:')
print('='*70)
for amb in sir['ambiguities']:
    conf = amb['confidence']
    flag = '⚠' if conf < 0.7 else '✓'
    print(f"{flag} [{conf:.2f}] {amb['description']}")
    print(f"   Assumption: {amb['primary_assumption']}")
    print(f"   Location: {amb['location']}")
    print()